<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=344741565" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 5 (FINAL FOLD): CUSTOM CNN =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42  # fixed globally across ALL folds and ALL architectures
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, MaxPooling2D, Dropout, GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 5
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print(f"Config loaded, CURRENT_FOLD = {CURRENT_FOLD} (FINAL FOLD)")

CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch"

def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print(f"Proportions: train {len(train_df)/len(cv_assignments)*100:.1f}% | val {len(val_df)/len(cv_assignments)*100:.1f}% | test {len(test_df)/len(cv_assignments)*100:.1f}%")
print("(expect roughly 68% / 12% / 20%, test size should equal fold 5's own size, 4,698)")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    f"STOP: FOLD {CURRENT_FOLD} LEAKAGE detected"
print(f"Fold {CURRENT_FOLD} leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

def make_fold_gens(preprocess_fn):
    if preprocess_fn is None:
        train_idg = ImageDataGenerator(rescale=1./255, **AUG)
        eval_idg  = ImageDataGenerator(rescale=1./255)
    else:
        train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
        eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_custom_cnn(num_classes=6, shape=(224,224,3)):
    return Sequential([
        Input(shape=shape),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(32,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(64,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        Conv2D(128,3,padding='same',activation='relu'), BatchNormalization(),
        MaxPooling2D(), Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256,activation='relu'), Dropout(0.5),
        Dense(num_classes,activation='softmax')
    ])

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Training Custom CNN. =====\n")

# ---------- CUSTOM CNN, FOLD 5 ----------
tr, va, te = make_fold_gens(None)
print("class_indices:", te.class_indices)

model = build_custom_cnn()
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom.keras', monitor='val_accuracy', save_best_only=True),
       CSVLogger(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom_log.csv', append=False)]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

y_true = np.asarray(te.classes)
y_prob = model.predict(te, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_custom.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom.keras')
verify_acc = reloaded.evaluate(te, verbose=0)[1]
live_acc = accuracy_score(y_true, y_pred)
print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
del reloaded; gc.collect(); tf.keras.backend.clear_session()

result_row = dict(fold=CURRENT_FOLD, arch='custom', accuracy=live_acc,
    macro_f1=f1_score(y_true,y_pred,average='macro'),
    macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
    macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
    macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true))

FOLD_ACC_CUSTOM = [0.47734326505276226, 0.563886606409203, 0.493689524610854, 0.5233824471492633]
prior_mean = np.mean(FOLD_ACC_CUSTOM)
deviation = abs(live_acc - prior_mean) * 100
flag = "  <-- FLAG: deviates >5pp from folds 1-4 mean" if deviation > 5 else "  (within normal range)"
print(f"\nFold {CURRENT_FOLD} vs Folds 1-4 mean: {live_acc:.4f} vs {prior_mean:.4f}, deviation {deviation:.1f}pp{flag}")

pd.DataFrame([result_row]).to_csv(f'/kaggle/working/cv_f{CURRENT_FOLD}_custom_result.csv', index=False)
print(f"\nRESULT: {result_row}")
print(f"Saved: /kaggle/working/cv_f{CURRENT_FOLD}_custom_result.csv")
print(">>> DOWNLOAD THIS FILE TO YOUR COMPUTER NOW. <<<")
print(f"\nStill needed for fold {CURRENT_FOLD} (LAST FOLD): eff, mob, res.")

2026-08-25 02:35:49.654484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787625349.855959      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787625349.911170      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787625350.379660      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787625350.379713      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787625350.379717      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded, CURRENT_FOLD = 5 (FINAL FOLD)
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

===== FOLD 5 SPLIT =====
Train 16,249 | Val 2,889 | Test 4,698
Proportions: train 68.2% | val 12.1% | test 19.7%
(expect roughly 68% / 12% / 20%, test size should equal fold 5's own size, 4,698)
Fold 5 leakage check: PASS
Fold 5 class_weight: {np.str_('bcc'): np.float64(1.235), np.str_('bkl'): np.float64(1.548), np.str_('df'): np.float64(16.314), np.str_('melanoma'): np.float64(0.859), np.str_('nevus'): np.float64(0.307), np.str_('vasc'): np.float64(15.3)}

===== Fold 5 setup verified. Training Custom CNN. =====

Found 16249 validated image filenames belonging to 6 classes.
Found 2889 validated image filenames belonging to 6 classes.
Found 4698 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1787625484.460131      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787625484.466151      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/60


E0000 00:00:1787625487.779999      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787625489.077044      66 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787625491.478815      66 service.cc:152] XLA service 0x7f22d9a1d310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787625491.478854      66 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787625491.478858      66 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787625491.757725      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


508/508 [==============================] - 695s 1s/step - loss: 1.7731 - accuracy: 0.3160 - val_loss: 1.6612 - val_accuracy: 0.2887
Epoch 2/60
508/508 [==============================] - 494s 973ms/step - loss: 1.6308 - accuracy: 0.3492 - val_loss: 2.0778 - val_accuracy: 0.1845
Epoch 3/60
508/508 [==============================] - 549s 1s/step - loss: 1.5595 - accuracy: 0.3527 - val_loss: 1.5322 - val_accuracy: 0.3818
Epoch 4/60
508/508 [==============================] - 496s 977ms/step - loss: 1.5034 - accuracy: 0.3994 - val_loss: 1.4079 - val_accuracy: 0.4829
Epoch 5/60
508/508 [==============================] - 495s 975ms/step - loss: 1.4902 - accuracy: 0.4041 - val_loss: 1.5970 - val_accuracy: 0.3679
Epoch 6/60
508/508 [==============================] - 496s 977ms/step - loss: 1.4726 - accuracy: 0.4065 - val_loss: 1.4997 - val_accuracy: 0.3859
Epoch 7/60
508/508 [==============================] - 491s 967ms/step - loss: 1.4224 - accuracy: 0.4150 - val_loss: 1.4706 - val_accuracy: 0.